## NOTE: might need to install all packages from starter code

^^^^^^^

In [11]:
# !pip uninstall -y vllm vllm-flash-attn flashinfer-python flashinfer-cubin humming-kernels tokenspeed-mla tokenspeed-triton quack-kernels

# !pip install -U pip

# !pip install "transformers<=4.57" sympy numpy tqdm bitsandbytes "antlr4-python3-runtime==4.11.1"

# !pip install "vllm==0.11.1" --extra-index-url https://download.pytorch.org/whl/cu128

In [12]:
# # Additional dependencies for SFT
# !pip install trl peft accelerate bitsandbytes datasetsndbytes datasets

In [13]:
import importlib, sys

packages = [
    "torch", "transformers", "peft", "trl",
    "bitsandbytes", "accelerate", "datasets",
    "vllm", "sympy", "numpy", "tokenizers",
]

print(f"Python: {sys.version}\n" + "-" * 50)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<20} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<20} NOT INSTALLED")

# Also check CUDA
try:
    import torch
    print(f"\n{'CUDA available':<20} {torch.cuda.is_available()}")
    print(f"{'CUDA version':<20} {torch.version.cuda}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}  |  {props.total_memory / 1e9:.1f} GB  |  SM {props.major}.{props.minor}")
except ImportError:
    pass

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
--------------------------------------------------
torch                2.9.0+cu128
transformers         4.56.2
peft                 0.19.1
trl                  1.5.1
bitsandbytes         0.49.2
accelerate           1.13.0
datasets             4.8.5
vllm                 0.11.1
sympy                1.14.0
numpy                2.0.2
tokenizers           0.22.2

CUDA available       True
CUDA version         12.8
  GPU 0: NVIDIA A100-SXM4-40GB  |  42.4 GB  |  SM 8.0


**Check that all of above are installed** ^^

## Dataset

In [14]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", name="default")
train_ds = ds["train"]

print(f"Loaded {len(train_ds)} examples")

# Remove MCQ examples
train_ds = train_ds.filter(lambda x: x["question_type"] != "MCQ")
print(f"After removing MCQ: {len(train_ds)} examples")

# Use only 50k examples to speed up training
train_ds = train_ds.shuffle(seed=42).select(range(50_000))
print(f"Using subset: {len(train_ds)} examples")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Loaded 93733 examples
After removing MCQ: 77773 examples
Using subset: 50000 examples


In [15]:
SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

def format_example(example):
    solution_with_answer = (
        example["solution"].rstrip()
        + f"\n\nFinal Answer: \\boxed{{{example['answer']}}}"
    )
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": example["problem"]},
            {"role": "assistant", "content": solution_with_answer},
        ]
    }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)

## Load Base Model

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,        # nested quantization → saves ~0.4 GB
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — optimal for normally distributed weights
    # bnb_4bit_compute_dtype=torch.float16,  # A30 lacks BF16 tensor cores — use fp16
    bnb_4bit_compute_dtype=torch.bfloat16,  # H100, A100, and newer support BF16 tensor cores
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # Required for gradient checkpointing
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"        # Required for SFT packing

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [17]:
# print(
#     tokenizer.apply_chat_template(
#         train_ds[0]["messages"],
#         tokenize=False
#     )
# )

## Attach LoRA Adapters

In [18]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # Casts LayerNorm to fp32, enables grad checkpointing

lora_config = LoraConfig(
    r=32,                          # Rank — higher = more capacity, more memory
    lora_alpha=16,                 # Scaling factor: effective LR ≈ lora_alpha / r
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1–3% of total params trainable

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


## Train Model

In [19]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./checkpoints/sft_qlora",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    gradient_checkpointing=True,
    optim="paged_adamw_32bit",

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # fp16=True, # for A30
    bf16=True,   # for H100, A100, and newer

    logging_steps=50,
    save_steps=500,
    save_strategy="steps",
    save_total_limit=2,

    report_to="none",

    max_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train() # if starting training from scratch
# trainer.train(resume_from_checkpoint=True) # if resuming training

trainer.save_model("./models/sft_qlora_adapter")
tokenizer.save_pretrained("./models/sft_qlora_adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


KeyboardInterrupt: 

## Save Model

In [1]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

!pip install --upgrade "torchao>=0.16.0"

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

peft_model = PeftModel.from_pretrained(base_model, "./models/sft_qlora_adapter")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained("./models/sft_merged", safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.save_pretrained("./models/sft_merged")

print("Merged model saved to ./models/sft_merged")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Merged model saved to ./models/sft_merged


In [2]:
from huggingface_hub import login

login()

In [3]:
from huggingface_hub import HfApi

repo_id = "benjiengee/qwen3-4b-thinking-sft-merged"

api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)

api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path="./models/sft_merged",
    commit_message="Upload merged SFT model",
)

print("Uploaded merged model to:", f"https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sft_merged/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...0001-of-00002.safetensors:   1%|1         | 56.0MB / 4.97GB            

  ...0002-of-00002.safetensors:   0%|          | 5.43MB / 3.08GB            

Uploaded merged model to: https://huggingface.co/benjiengee/qwen3-4b-thinking-sft-merged


Then in `run_inference.py`, you would load: `MODEL_ID = "mrmsantos03/qwen3-4b-thinking-sft-merged"`

In [4]:
from huggingface_hub import HfApi

repo_id = "benjiengee/qwen3-4b-thinking-sft-adapter"

api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)

api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path="./models/sft_qlora_adapter",
    commit_message="Upload QLoRA adapter",
)

print("Uploaded adapter to:", f"https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ra_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:   0%|          | 61.4kB /  132MB            

  ...adapter/training_args.bin:   2%|1         |   118B / 6.22kB            

Uploaded adapter to: https://huggingface.co/benjiengee/qwen3-4b-thinking-sft-adapter


Fine-tuned model weights are hosted on Hugging Face at:
https://huggingface.co/mrmsantos03/qwen3-4b-thinking-sft-merged

The inference script loads this model inside `run_inference()`.